# Notebook 03f — Modelo combinado: Tipo de tumor + Mutaciones driver + Tratamientos

**Objetivo**: comprobar si añadir las variables de tratamiento encima de las mutaciones mejora el R².  
**Target**: `log1p(Tumor_Telomere fusion rate)`  
**Features**:
- `Mixed_def_*` — tipo de tumor en formato binario (58 columnas)
- Genes driver — flag binario por gen (solo genes con ≥10 pacientes, para evitar ruido)
- Variables verdes — firstResponse, treatment_agrupado, Mesenchimal_origin
- Variables azules — biopsySite, hasSystemicPreTreatment, hasRadiotherapyPreTreatment, treatmentGiven, treatment1

**Referencia**: el mejor R² hasta ahora fue 0.085 con Lasso en el notebook 03c (solo tratamientos).  
**Pregunta**: ¿añadir los genes mejora ese número?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import re

from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import scipy.stats as stats

import os
os.chdir('..')  

print('Librerías cargadas OK')

## 1. Carga de datos base

In [ ]:
TARGET = 'Tumor_Telomere fusion rate (Gt-end/coverage*purity)'

# Base: necesitamos cancer_type_code para el join mesenquimal
df = pd.read_csv('datos_limpios_modelo_a.csv',
                 usecols=['Patient_ID_IF_mod', TARGET, 'cancer_type_code'])
print(f'Base: {df.shape}')

# Tipos de tumor (Mixed_def_*) — vienen de datos_modelo_final.csv
df_final = pd.read_csv('datos_modelo_final.csv')
mixed_cols = [c for c in df_final.columns if c.startswith('Mixed_def_')]
df_mixed = df_final[['Patient_ID_IF_mod'] + mixed_cols]
df = df.merge(df_mixed, on='Patient_ID_IF_mod', how='left')
print(f'Tras añadir tipos de tumor: {df.shape}')

## 2. Variables de tratamiento

In [ ]:
# Variables verdes y azules desde revision_color_code.xlsx
df_trat = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                        usecols=['Patient_ID_IF_mod', 'firstResponse', 'treatment',
                                 'biopsySite', 'hasSystemicPreTreatment',
                                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1'])
df = df.merge(df_trat, on='Patient_ID_IF_mod', how='left')
print(f'Tras añadir tratamientos: {df.shape}')

# Variable mesenquimal
df_mes = pd.read_excel('Tumor_mesenchimal.xlsx')
df_mes.columns = ['cancer_type_code', 'Mesenchimal_origin']
df = df.merge(df_mes, on='cancer_type_code', how='left')
print(f'Tras añadir mesenquimal: {df.shape}')

## 3. Preparación de variables de tratamiento

In [ ]:
# NaN → Sin_dato (conservamos todos los pacientes)
df['firstResponse'] = df['firstResponse'].fillna('Sin_dato')
df['Mesenchimal_origin'] = df['Mesenchimal_origin'].fillna('Sin_dato')
df['hasSystemicPreTreatment'] = df['hasSystemicPreTreatment'].fillna('Sin_dato')
df['hasRadiotherapyPreTreatment'] = df['hasRadiotherapyPreTreatment'].fillna('Sin_dato')
df['treatmentGiven'] = df['treatmentGiven'].fillna('Sin_dato')

# treatment: top 15 + Otro
top15 = df['treatment'].value_counts().head(15).index.tolist()
df['treatment_agrupado'] = df['treatment'].apply(
    lambda x: x if x in top15 else ('Sin_dato' if pd.isna(x) else 'Otro')
)

# biopsySite: top 10 + Otro
top10_biopsy = df['biopsySite'].value_counts().head(10).index.tolist()
df['biopsySite_agrupado'] = df['biopsySite'].apply(
    lambda x: x if x in top10_biopsy else ('Sin_dato' if pd.isna(x) else 'Otro')
)

# treatment1: top 10 + Otro
top10_t1 = df['treatment1'].value_counts().head(10).index.tolist()
df['treatment1_agrupado'] = df['treatment1'].apply(
    lambda x: x if x in top10_t1 else ('Sin_dato' if pd.isna(x) else 'Otro')
)

print('Variables de tratamiento preparadas OK')

## 4. Matriz de mutaciones (genes con ≥10 pacientes)

In [ ]:
df_drivers = pd.read_excel(
    'Drivers_TF_TVR_TL_to_ALV.xlsx',
    sheet_name='Sheet1',
    header=0,
    usecols=['Patient_ID_IF_mod', 'gene']
)

# Filtro ≥10 pacientes por gen (evita genes raros con poco soporte estadístico)
gene_counts = df_drivers.groupby('gene')['Patient_ID_IF_mod'].nunique()
genes_validos = gene_counts[gene_counts >= 10].index.tolist()
print(f'Genes con >= 10 pacientes: {len(genes_validos)} (de {df_drivers["gene"].nunique()} totales)')

df_drivers = df_drivers[df_drivers['gene'].isin(genes_validos)].drop_duplicates(
    subset=['Patient_ID_IF_mod', 'gene']
)

# Pivot: formato largo → formato ancho (paciente × gen)
df_genes = df_drivers.assign(valor=1).pivot(
    index='Patient_ID_IF_mod', columns='gene', values='valor'
).fillna(0).astype(int)
df_genes.columns = ['gene_' + re.sub(r'[^A-Za-z0-9_]', '_', col) for col in df_genes.columns]
df_genes = df_genes.reset_index()
print(f'Matriz de mutaciones: {df_genes.shape[0]} pacientes × {df_genes.shape[1]-1} genes')

## 5. Unión final de todos los datasets

In [ ]:
df = df.merge(df_genes, on='Patient_ID_IF_mod', how='inner')
print(f'Dataset final: {df.shape}')
print(f'Pacientes: {df.shape[0]}')
print(f'Nulos en target: {df[TARGET].isna().sum()}')

## 6. Target — log1p

In [ ]:
y = np.log1p(df[TARGET])
print(f'Skewness original: {df[TARGET].skew():.2f}')
print(f'Skewness tras log1p: {y.skew():.2f}')

## 7. Definición de features

In [ ]:
gene_cols = [c for c in df.columns if c.startswith('gene_')]

cols_tratamiento = ['firstResponse', 'treatment_agrupado', 'Mesenchimal_origin',
                    'biopsySite_agrupado', 'hasSystemicPreTreatment',
                    'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1_agrupado']

# Dummies para las variables categóricas de tratamiento
X_trat = pd.get_dummies(df[cols_tratamiento], drop_first=True)

# Combinamos todo
X = pd.concat([df[mixed_cols].reset_index(drop=True),
               df[gene_cols].reset_index(drop=True),
               X_trat.reset_index(drop=True)], axis=1)

print(f'Features totales: {X.shape[1]}')
print(f'  - Tipo de tumor (Mixed_def_*): {len(mixed_cols)}')
print(f'  - Genes driver (>=10 pac.):    {len(gene_cols)}')
print(f'  - Variables de tratamiento:    {X_trat.shape[1]}')
print(f'Shape final → X: {X.shape} | y: {y.shape}')

## 8. Split train / test + escalado

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

## 9. Función de evaluación

In [ ]:
def evaluar_modelo(nombre, modelo, X_tr, X_te, y_tr, y_te, feature_names):
    modelo.fit(X_tr, y_tr)
    y_pred_log  = modelo.predict(X_te)
    # Deshacer log1p → métricas en espacio original
    y_pred_orig = np.expm1(y_pred_log)
    y_te_orig   = np.expm1(np.array(y_te))

    r2   = r2_score(y_te_orig, y_pred_orig)
    mae  = mean_absolute_error(y_te_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_te_orig, y_pred_orig))
    cv   = cross_val_score(modelo, X_tr, y_tr, cv=5, scoring='r2').mean()

    sep = '=' * 55
    print(sep)
    print(f'  {nombre}')
    print(sep)
    print(f'  R²  (test, espacio original):  {r2:.4f}')
    print(f'  R²  (CV-5 train):              {cv:.4f}')
    print(f'  MAE  (espacio original):        {mae:.4f}')
    print(f'  RMSE (espacio original):        {rmse:.4f}')

    coefs = pd.Series(modelo.coef_, index=feature_names)
    nz = coefs[coefs != 0].sort_values(key=abs, ascending=False)
    print(f'  Coefs no nulos: {len(nz)} / {len(coefs)}')
    print('  Top 10:')
    print(nz.head(10).to_string())
    print()

    return {'nombre': nombre, 'modelo': modelo, 'R2_test': r2, 'R2_cv': cv,
            'MAE': mae, 'RMSE': rmse, 'y_pred': y_pred_orig, 'coefs': coefs}

## 10. Lasso

In [ ]:
res_lasso = evaluar_modelo(
    'Lasso (alpha=0.001)', Lasso(alpha=0.001, max_iter=10000, random_state=42),
    X_train_sc, X_test_sc, y_train, y_test, X.columns
)

## 11. Ridge

In [ ]:
res_ridge = evaluar_modelo(
    'Ridge (alpha=1.0)', Ridge(alpha=1.0, random_state=42),
    X_train_sc, X_test_sc, y_train, y_test, X.columns
)

## 12. Elastic Net

In [ ]:
res_enet = evaluar_modelo(
    'Elastic Net (alpha=0.001, l1_ratio=0.5)',
    ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42),
    X_train_sc, X_test_sc, y_train, y_test, X.columns
)

## 13. Comparativa final

In [ ]:
resultados = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('modelo', 'y_pred', 'coefs')}
    for r in [res_lasso, res_ridge, res_enet]
]).set_index('nombre')

print('=== COMPARATIVA FINAL — TODO COMBINADO ===')
print(resultados.round(4).to_string())
print()
print('--- Referencia notebooks anteriores ---')
print('03c Lasso (solo tratamientos):          R2_test=0.0846')
print('03e Lasso (solo tipo tumor + genes):    R2_test=0.0336')
print()
mejor = resultados['R2_test'].idxmax()
print(f'Mejor modelo este notebook: {mejor}')

## 14. Genes seleccionados por Lasso

In [ ]:
coefs_lasso = res_lasso['coefs']
genes_sel = coefs_lasso[
    (coefs_lasso != 0) & (coefs_lasso.index.str.startswith('gene_'))
].sort_values(key=abs, ascending=False)

print(f'Genes seleccionados por Lasso: {len(genes_sel)} de {len(gene_cols)}')
print()
print('Positivos (más fusiones):')
for g, c in genes_sel[genes_sel > 0].items():
    print(f'  {g.replace("gene_", ""):20s}  coef={c:+.4f}')
print()
print('Negativos (menos fusiones):')
for g, c in genes_sel[genes_sel < 0].items():
    print(f'  {g.replace("gene_", ""):20s}  coef={c:+.4f}')

## 15. Gráficos real vs predicho

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, res in zip(axes, [res_lasso, res_ridge, res_enet]):
    y_pred = np.array(res['y_pred'])
    y_real = np.expm1(np.array(y_test))
    ax.scatter(y_real, y_pred, alpha=0.3, s=10, color='steelblue')
    lim = max(y_real.max(), y_pred.max())
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1, label='Perfecto')
    ax.set_xlabel('Real (espacio original)')
    ax.set_ylabel('Predicho (espacio original)')
    ax.set_title(f"{res['nombre']}
R²={res['R2_test']:.4f}")
    ax.legend(fontsize=8)

plt.suptitle('Todo combinado: tumor + genes + tratamientos', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('notebooks/img_03f_todo_combinado.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img_03f_todo_combinado.png')